# Renters (specialty-ltv) v1.5.0 - loss ratio refit validation (cat excluded)Same structure as the classic notebook, scoped to line 71.## VERIFY BEFORE RUNNINGCells marked `# >>> VERIFY` contain names I could not confirm from what you haveshown me. Fix them first; a wrong path here produces a clean run against thewrong data, which is worse than a crash.Open items carried into this notebook:1. **`nt0` branch shape.** Nick's notebook treats NT0 as a single flat value   (`nt0_lr`). Production has `nt0_slope` + `nt0_int`. For line 16 both agree   (`nt0_slope=0`, flat at 1.0057). For the mirrored lines the two conventions   differ at nt6=1 by `slope_yr * 0.5` — about 0.007 on line 32. `FLAT_NT0`   below toggles between them. Confirm which the scorer implements.   (Note: mirroring `nt0_slope=slope_yr` is safe under *either* convention,   since it reproduces the renewal curve.)2. **Nick's notebook fits disagree with production** on lines 78 and 88, and it   carries commented-out variants. It is a different vintage. `PROD_FITS` here   is transcribed from `1_policy.py`, not from his notebook.3. **Cat add-back** is still unresolved. Section C is the test for it.

In [ ]:
%env ENV_FOR_DYNACONF = prod%env DYNACONF_GIT_BRANCH = feature/B-2895893%env DYNACONF_GIT_CHECKOUT = feature/B-2895893

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as plt# >>> VERIFY these import paths against specialty_ltv.import ltv_helpers.non_spark_helpers as nsfrom specialty_ltv import paths as p%matplotlib inlinepd.options.display.max_rows = 2000pd.options.display.max_columns = 500

In [ ]:
# specialty-ltv's dodo.py does NOT expose a 'score_internal_results_unbalanced'.# calculate_ltv_unbalanced writes p.agg_for_balance and p.balance_factors;# calculate_ltv_balanced writes p.score_internal_results.# >>> VERIFY which of these actually carries per-policy loss / cat / premium.P_SCORED_NEW = p.score_internal_resultsP_BALANCE_FACTORS = p.balance_factorsP_AGG_FOR_BALANCE = p.agg_for_balanceP_SCORED_PRIOR = ("tmx-smsiweb/specialty-ltv/prod/"                  "<PRIOR_RELEASE>/score/score_internal_results/")  # >>> VERIFYprint(P_SCORED_NEW)

## Section A - fits

In [ ]:
# Cat-excluded refit. slope_yr / intercept at 10 dp from the notebook's# other_lr_tuple print; nt0_int at 6 dp from spl_fits (only precision available).# >>> VERIFY precision convention before merging to 1_policy.py.NEW_FITS = {    # line: (slope_yr,       intercept,      nt0_slope,      nt0_int)    "16": (-0.0341583943,  0.6105496031,  0.0,            0.970097),    "32": (-0.0133350880,  0.3664844453, -0.0133350880,   0.3664844453),    "71": (-0.0435713887,  0.6252041479, -0.0435713887,   0.6252041479),    "72": ( 0.0,           0.3154266994,  0.0,            0.3154266994),    "78": ( 0.0,           0.4930239249,  0.0,            0.4930239249),    "88": ( 0.0,           0.7888171791,  0.0,            1.180678),    "90": (-0.0237481671,  0.5650788252,  0.0,            0.769902),}# Shipped fits, transcribed from classic_spl_ltv/jobs/score/1_policy.pyPROD_FITS = {    "16": (-0.02294,   0.6347,   0.0,       1.0057),    "32": (-0.01407,   0.6206,  -0.01407,   0.6206),    "71": (-0.05045,   0.7259,  -0.05045,   0.7259),    "72": (-0.003477,  0.6060,  -0.003477,  0.6060),    "78": (-0.03717,   0.82678, -0.03717,   0.82678),    "88": ( 0.0,       0.7571,   0.0,       1.2079),    "90": (-0.02329,   0.7527,   0.0,       0.9611),}LINE_NAMES = {"16": "Specialty Auto", "32": "Mfg Home", "71": "Renters",              "72": "Landlord", "78": "Condo", "88": "PUP", "90": "Boat"}

In [ ]:
# Renters is line 71 only. Note that 71 also appears in classic's spl_lr_fits -# resolve which repo actually scores it before trusting either run.RENTERS_LINES = ["71"]  # >>> VERIFYNEW_FITS = {k: v for k, v in NEW_FITS.items() if k in RENTERS_LINES}PROD_FITS = {k: v for k, v in PROD_FITS.items() if k in RENTERS_LINES}NEW_FITS, PROD_FITS

In [ ]:
# Fit evaluation, matching Nick's create_slope_intercept_spl_lr_dfFLAT_NT0 = True  # >>> VERIFY  True = Nick's flat nt0_lr; False = nt0_int + nt0_slope*ntrNT6_CAP = 18  # curve is capped at NTR = 9 (the censored bucket)def eval_fit(nt6, f):    """f = (slope_yr, intercept, nt0_slope, nt0_int). Returns loss ratio."""    slope_yr, intercept, nt0_slope, nt0_int = f    ntr = min(nt6, NT6_CAP) / 2    if nt6 < 2:        return nt0_int if FLAT_NT0 else nt0_int + nt0_slope * ntr    return intercept + slope_yr * ntrdef fit_table(fits, max_nt6=19):    rows = []    for line, f in sorted(fits.items()):        for nt6 in range(max_nt6 + 1):            rows.append(                {"drv_line": str(line), "nt6": nt6, "ntr": min(nt6, NT6_CAP) / 2,                 "lr_expected": eval_fit(nt6, f)}            )    return pd.DataFrame(rows)

### Fit-level delta

In [ ]:
# Fit-level delta. No pipeline data needed - pure arithmetic on the two dicts.rows = []for line in sorted(NEW_FITS):    for nt6 in [0, 2, 4, 6, 10, 18]:        rows.append({            "drv_line": line, "name": LINE_NAMES.get(line, ""), "ntr": min(nt6, NT6_CAP) / 2,            "prod": eval_fit(nt6, PROD_FITS[line]),            "new": eval_fit(nt6, NEW_FITS[line]),        })delta = pd.DataFrame(rows)delta["diff"] = delta["new"] - delta["prod"]delta.pivot_table(index=["drv_line", "name"], columns="ntr", values="diff").round(4)

### Balance factors provenance

In [ ]:
# Yinan's one hard constraint: the balance step must NOT fall back to a prior# version's file. Confirm balance_factors was written by THIS run.bf = ns.read_parquet_s3_to_pandas(P_BALANCE_FACTORS)print(len(bf))bf.head(20)

## Section B - propagation check (weak, but necessary)Confirms the scorer picked up the new fits: every policy's `lr` should equalthe fit evaluated at its `nt6`.This proves the numbers reached the pipeline. It proves nothing about whetherthey are right - `loss = premium * lr` by construction, so dividing back iscircular. Same trap as the expense-ratio validation.

In [ ]:
df = ns.read_parquet_s3_to_pandas(P_SCORED_NEW)df["drv_line"] = df["drv_line"].astype(str)df = df[df["drv_line"].isin(NEW_FITS)]df["lr_expected"] = [eval_fit(n, NEW_FITS[l])                     for n, l in zip(df["nt6"], df["drv_line"])]df["lr_gap"] = (df["lr"] - df["lr_expected"]).abs()chk = df.groupby("drv_line", as_index=False).agg(    n=("lr", "size"), max_gap=("lr_gap", "max"), mean_lr=("lr", "mean"))chk["pass"] = chk["max_gap"] < 1e-6chk.round(6)

## Section C - cat add-back / balancing absorption`loss` and `cat` are separate columns, so:- `lr_ex_cat = loss / premium`- `lr_total  = (loss + cat) / premium`But `score_internal_results` is POST-balancing. If `process_lr_balance` rescalesto a Finance target, `lr_total` being preserved proves nothing - the balance stepwould absorb the cat strip either way.So the discriminating column is `lr_balance_amt`. Diff it across releases:- **`d_bal` roughly cancels `d_cat`** -> balancing swallowed the change. The  -$784M is a pre-balance artifact and does not reach the P&L.- **`d_bal` is flat while `lr_ex_cat` drops** -> the change is real and nothing  downstream put cat back.Either way, read `cat_load_preprocess` in `dodo_finance.py` before presenting.Renters cat share was 16.7%, so the ex-cat drop should be visible but farsmaller than Landlord or Mfg Home. If it isn't, the fits did not land.

In [ ]:
# >>> VERIFY both paths. PRIOR must point at the previous release's output,# not at your branch.df_new = ns.read_parquet_s3_to_pandas(P_SCORED_NEW)df_old = ns.read_parquet_s3_to_pandas(P_SCORED_PRIOR)def lr_summary(df, tag):    d = df.copy()    d["drv_line"] = d["drv_line"].astype(str)    g = d.groupby("drv_line", as_index=False).agg(        premium=("premium", "sum"), loss=("loss", "sum"), cat=("cat", "sum"),        bal=("lr_balance_amt", "sum"),    )    g["lr_ex_cat"] = g["loss"] / g["premium"]    g["lr_total"] = (g["loss"] + g["cat"]) / g["premium"]    g["cat_share"] = g["cat"] / (g["loss"] + g["cat"])    g["src"] = tag    return gcmp = lr_summary(df_new, "new").merge(    lr_summary(df_old, "prior"), on="drv_line", suffixes=("_new", "_old"))cmp["d_ex_cat"] = cmp["lr_ex_cat_new"] - cmp["lr_ex_cat_old"]cmp["d_total"] = cmp["lr_total_new"] - cmp["lr_total_old"]cmp["d_premium"] = cmp["premium_new"] - cmp["premium_old"]cmp["d_bal"] = cmp["bal_new"] - cmp["bal_old"]cmp["d_cat"] = cmp["cat_new"] - cmp["cat_old"]cmp[["drv_line", "lr_ex_cat_old", "lr_ex_cat_new", "d_ex_cat",     "lr_total_old", "lr_total_new", "d_total", "d_premium",     "d_cat", "d_bal"]].round(2)# THE TEST: if d_bal absorbs roughly -d_cat, the balance step swallowed the# cat strip and the -$784M never reaches the P&L. Compare the two directly.

### Dollar impact

In [ ]:
# Dollar impact, premium-weighted. This is the headline for the Loop page.# The unweighted mean over an NTR grid in Nick's In[26] is NOT comparable to a# scored average - do not use it.cmp["dollar_impact_ex_cat"] = cmp["d_ex_cat"] * cmp["premium_new"]tot_prem = cmp["premium_new"].sum()tot_impact = cmp["dollar_impact_ex_cat"].sum()out = cmp[["drv_line", "premium_new", "d_ex_cat", "dollar_impact_ex_cat"]].copy()out["pct_of_impact"] = out["dollar_impact_ex_cat"] / tot_impactprint(f"total premium: {tot_prem:,.0f}")print(f"total impact:  {tot_impact:,.0f}")print(f"points:        {tot_impact / tot_prem * 100:.1f}")out.sort_values("dollar_impact_ex_cat").round(4)

## Section E - remaining verificationSmall, cheap, and all still open.

In [ ]:
# 1. Cat code '9' - is it a real PCS-style serial or an internal sentinel?#    A true catastrophe clusters in month x state. A sentinel does not.#    >>> Run this against the CLAIMS extract, not the scored output.## claims["CATCD"].value_counts(dropna=False)# pd.crosstab(claims.loc[claims["CATCD"] == "9", "ACTMO"],#             claims.loc[claims["CATCD"] == "9", "GEOST"])# 2. PUP zero cat - distinguish the two failure modes.#    Is CATCD populated-and-all-blank for line 88, or absent entirely?## claims.loc[claims["ALINE"] == "88", "CATCD"].isna().mean()# claims.loc[claims["ALINE"] == "88", "CATCD"].notna().sum()# 3. Exposure window control. Rerun the fits on 2023-2024 only.#    ACTYR 125 is immature; if losses are reported-to-date rather than developed,#    every level here is biased low and not comparable to the prior analysis.#    This is a control, not a decision - run it regardless.# 4. Landlord positive slope (+0.00733, p<0.001, $1.6B) suppressed by the#    'slope > 0 -> flat' rule. Refit with NTR=9 dropped:#      - positive only WITH the censored bucket -> rule is accidentally right#      - positive WITHOUT it -> the rule is suppressing real signal on the#        single largest line#    Cat removal strips the fattest tail from the residuals, so the post-cat#    p-value is MORE credible than any pre-cat one. That argues for the slope.